# Human-in-the-Loop Workflow gamit ang Microsoft Agent Framework

## 🎯 Mga Layunin sa Pagkatuto

Sa notebook na ito, matututunan mo kung paano ipatupad ang **human-in-the-loop** workflows gamit ang `RequestInfoExecutor` ng Microsoft Agent Framework. Ang makapangyarihang pattern na ito ay nagbibigay-daan para ihinto ang mga AI workflow upang mangolekta ng input mula sa tao, na ginagawang interaktibo ang iyong mga agent at nagbibigay kontrol sa mga tao sa mga kritikal na desisyon.

## 🔄 Ano ang Human-in-the-Loop?

Ang **Human-in-the-loop (HITL)** ay isang design pattern kung saan ang mga AI agent ay tumitigil sa pagpapatupad upang humingi ng input mula sa tao bago magpatuloy. Mahalaga ito para sa:

- ✅ **Kritikal na desisyon** - Kumuha ng pahintulot mula sa tao bago isagawa ang mahahalagang aksyon
- ✅ **Malabong sitwasyon** - Hayaan ang tao na linawin kapag hindi sigurado ang AI
- ✅ **Mga kagustuhan ng gumagamit** - Tanungin ang mga gumagamit na pumili sa pagitan ng mga maraming opsyon
- ✅ **Pagsunod at kaligtasan** - Siguraduhin ang pagsubaybay ng tao sa mga pinamamahalaang operasyon
- ✅ **Interaktibong karanasan** - Bumuo ng mga conversational agent na tumutugon sa input ng gumagamit

## 🏗️ Paano Ito Gumagana sa Microsoft Agent Framework

Nagbibigay ang framework ng tatlong pangunahing bahagi para sa HITL:

1. **`RequestInfoExecutor`** - Isang espesyal na executor na humihinto sa workflow at nagpapalabas ng `RequestInfoEvent`
2. **`RequestInfoMessage`** - Base class para sa mga typed request payload na ipinapadala sa mga tao
3. **`RequestResponse`** - Nag-uugnay ng mga tugon ng tao sa mga orihinal na request gamit ang `request_id`

**Workflow Pattern:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Ang Ating Halimbawa: Pag-book ng Hotel na may Kumpirmasyon mula sa Gumagamit

Magpapatuloy tayo mula sa conditional workflow sa pamamagitan ng pagdaragdag ng kumpirmasyon ng tao **bago** magmungkahi ng mga alternatibong destinasyon:

1. Humihiling ang gumagamit ng isang destinasyon (e.g., "Paris")
2. Sinusuri ng `availability_agent` kung may bakanteng kuwarto
3. **Kung walang kuwarto** → tinatanong ng `confirmation_agent` "Gusto mo bang makita ang mga alternatibo?"
4. **Humihinto** ang workflow gamit ang `RequestInfoExecutor`
5. **Tumutugon ang tao** ng "oo" o "hindi" sa pamamagitan ng console input
6. Ang `decision_manager` ay nagro-route base sa tugon:
   - **Oo** → Ipakita ang mga alternatibong destinasyon
   - **Hindi** → Kanselahin ang kahilingan sa pag-book
7. Ipakita ang panghuling resulta

Ipinapakita nito kung paano bigyan ang mga gumagamit ng kontrol sa mga mungkahi ng agent!

---

Magsimula na tayo! 🚀


## Hakbang 1: Mag-import ng Mga Kailangan na Library

Ini-import natin ang mga karaniwang bahagi ng Agent Framework plus **mga klase na espesyal para sa human-in-the-loop**:
- `RequestInfoExecutor` - Tagapagpatupad na humihinto sa workflow para sa input ng tao
- `RequestInfoEvent` - Kaganapan na inilalabas kapag hinihingi ang input ng tao
- `RequestInfoMessage` - Batayang klase para sa mga naka-type na request payloads
- `RequestResponse` - Nag-uugnay ng mga sagot ng tao sa mga kahilingan
- `WorkflowOutputEvent` - Kaganapan para sa pagtukoy ng mga output ng workflow


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Hakbang 2: Tukuyin ang mga Pydantic Model para sa Nakabalangkas na Output

Itong mga modelong ito ang tumutukoy sa **iskema** na ibabalik ng mga ahente. Pinananatili namin ang lahat ng mga modelo mula sa conditional workflow at nagdagdag:

**Bago para sa Human-in-the-Loop:**
- `HumanFeedbackRequest` - Subclass ng `RequestInfoMessage` na naglalarawan sa request payload na ipinapadala sa mga tao
  - Naglalaman ng `prompt` (tanong na itatanong) at `destination` (konteksto tungkol sa hindi magagamit na lungsod)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Hakbang 3: Gumawa ng Kasangkapan sa Pag-book ng Hotel

Parehong kasangkapan mula sa conditional workflow - tinitingnan kung may available na mga kuwarto sa destinasyon.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Hakbang 4: Tukuyin ang mga Condition Functions para sa Routing

Kailangan natin ng **apat na condition functions** para sa ating human-in-the-loop na workflow:

**Mula sa conditional workflow:**
1. `has_availability_condition` - Nagro-route kapag MAY availability ang mga hotel
2. `no_availability_condition` - Nagro-route kapag WALANG availability ang mga hotel

**Bago para sa human-in-the-loop:**
3. `user_wants_alternatives_condition` - Nagro-route kapag ang user ay nagsabi ng "oo" sa mga alternatibo
4. `user_declines_alternatives_condition` - Nagro-route kapag ang user ay nagsabi ng "hindi" sa mga alternatibo


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Hakbang 5: Lumikha ng Decision Manager Executor

Ito ang **puso ng human-in-the-loop pattern**! Ang `DecisionManager` ay isang custom na `Executor` na:

1. **Tumatanggap ng feedback mula sa tao** sa pamamagitan ng `RequestResponse` na mga object
2. **Pinoproseso ang desisyon ng user** (oo/hindi)
3. **Niruruta ang workflow** sa pamamagitan ng pagpapadala ng mga mensahe sa mga angkop na ahente

Mga pangunahing tampok:
- Gumagamit ng `@handler` na dekorador upang ipakita ang mga metodo bilang mga hakbang sa workflow
- Tumatanggap ng `RequestResponse[HumanFeedbackRequest, str]` na naglalaman ng parehong orihinal na kahilingan at sagot ng user
- Naglalabas ng simpleng "oo" o "hindi" na mga mensahe na nagpapagana sa aming mga kondisyong function


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Hakbang 6: Gumawa ng Pasadyang Display Executor

Parehong display executor mula sa conditional workflow - nagbibigay ng panghuling resulta bilang output ng workflow.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Hakbang 7: I-load ang mga Environment Variables

I-configure ang LLM client (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Hakbang 8: Gumawa ng Mga AI Agent at Executor

Gumagawa tayo ng **anim na bahagi ng workflow**:

**Mga Agent (nakapaloob sa AgentExecutor):**
1. **availability_agent** - Sinusuri ang availability ng hotel gamit ang tool
2. **confirmation_agent** - 🆕 Naghahanda ng kahilingan para sa kumpirmasyon ng tao
3. **alternative_agent** - Nagsusuggest ng alternatibong mga lungsod (kapag sumagot ng oo ang user)
4. **booking_agent** - Hinihikayat ang pag-book (kapag may mga available na kwarto)
5. **cancellation_agent** - 🆕 Humahawak ng mensahe ng kanselasyon (kapag sumagot ng hindi ang user)

**Mga Espesyal na Executor:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` na nagpapatigil sa workflow para sa input ng tao
7. **decision_manager** - 🆕 Custom na executor na nagruruta batay sa tugon ng tao (naidefine na sa itaas)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Hakbang 9: Buuhin ang Workflow na may Human-in-the-Loop

Ngayon ay bubuuin natin ang workflow graph na may **conditional routing** kabilang ang human-in-the-loop path:

**Istruktura ng Workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Pangunahing Mga Koneksyon:**
- `availability_agent → confirmation_agent` (kapag walang mga kwarto)
- `confirmation_agent → prepare_human_request` (pagbabago ng uri)
- `prepare_human_request → request_info_executor` (pahinga para sa tao)
- `request_info_executor → decision_manager` (palagi - nagbibigay ng RequestResponse)
- `decision_manager → alternative_agent` (kapag ang user ay nagsabi ng "oo")
- `decision_manager → cancellation_agent` (kapag ang user ay nagsabi ng "hindi")
- `availability_agent → booking_agent` (kapag may mga kwarto)
- Lahat ng mga landas ay nagtatapos sa `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Hakbang 10: Patakbuhin ang Test Case 1 - Lungsod NA WALANG Availability (Paris na may Human Confirmation)

Ipinapakita ng test na ito ang **buong human-in-the-loop cycle**:

1. Humiling ng hotel sa Paris
2. sinusuri ng availability_agent → Walang mga kuwarto
3. lumilikha ang confirmation_agent ng tanong para sa tao
4. pinapahinto ng request_info_executor **ang workflow** at naglalabas ng `RequestInfoEvent`
5. **Nadidiskubre ng Aplikasyon ang event at pinaprompt ang user sa console**
6. Nagta-type ang User ng "yes" o "no"
7. Pinapadala ng Aplikasyon ang sagot gamit ang `send_responses_streaming()`
8. nagro-route ang decision_manager base sa sagot
9. Ipinapakita ang panghuling resulta

**Pangunahing Pattern:**
- Gamitin ang `workflow.run_stream()` para sa unang pag-ikot
- Gamitin ang `workflow.send_responses_streaming(pending_responses)` para sa mga sumunod na pag-ikot
- Makinig sa `RequestInfoEvent` upang malaman kung kailangan ang input ng tao
- Makinig sa `WorkflowOutputEvent` upang makuha ang panghuling mga resulta


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Hakbang 11: Patakbuhin ang Test Case 2 - Lungsod NA MAY Availability (Stockholm - Walang Kailangan na Human Input)

Ipinapakita ng test na ito ang **direktang daan** kapag may mga bakanteng kuwarto:

1. Humiling ng hotel sa Stockholm
2. sinisiyasat ng availability_agent → May bakanteng kuwarto ✅
3. nagmumungkahi ang booking_agent ng booking
4. ipinapakita ng display_result ang kumpirmasyon
5. **Walang kailangan na input mula sa tao!**

Direktang nilalaktawan ng workflow ang human-in-the-loop na proseso kapag may mga bakanteng kuwarto.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Mga Pangunahing Punto at Pinakamahusay na Praktis sa Human-in-the-Loop

### ✅ Mga Natutunan Mo:

#### 1. **RequestInfoExecutor Pattern**
Ang human-in-the-loop pattern sa Microsoft Agent Framework ay gumagamit ng tatlong pangunahing bahagi:
- `RequestInfoExecutor` - Humihinto sa workflow at naglalabas ng mga event
- `RequestInfoMessage` - Base class para sa mga typed na request payload (mag-subclass dito!)
- `RequestResponse` - Nagkokorelasyon ng mga tugon ng tao sa orihinal na mga kahilingan

**Mahalagang Pag-unawa:**
- Hindi nangongolekta ng input ang `RequestInfoExecutor` mismo - humihinto lamang ito sa workflow
- Kailangang makinig ang iyong application code sa `RequestInfoEvent` at mangolekta ng input
- Dapat mong tawagan ang `send_responses_streaming()` na may dict na nagmamapa ng `request_id` sa sagot ng user

#### 2. **Streaming Execution Pattern**
```python
# Unang pag-ulit
stream = workflow.run_stream(initial_request)

# Mga sumunod na pag-ulit (pagkatapos ng input ng tao)
stream = workflow.send_responses_streaming(pending_responses)

# Palaging iproseso ang mga kaganapan
events = [event async for event in stream]
```

#### 3. **Event-Driven Architecture**
Makinig sa mga espesipikong event para kontrolin ang workflow:
- `RequestInfoEvent` - Kailangan ang input ng tao (workflow humihinto)
- `WorkflowOutputEvent` - Available na ang huling resulta (workflow kumpleto na)
- `WorkflowStatusEvent` - Mga pagbabago sa estado (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, atbp.)

#### 4. **Custom Executors na may @handler**
Ipinapakita ng `DecisionManager` kung paano gumawa ng mga executors na:
- Gumagamit ng `@handler` decorator para i-expose ang mga method bilang mga hakbang ng workflow
- Tumanggap ng typed na mga mensahe (hal., `RequestResponse[HumanFeedbackRequest, str]`)
- Mag-route ng workflow sa pamamagitan ng pagpapadala ng mga mensahe sa ibang executors
- Mag-access ng context gamit ang `WorkflowContext`

#### 5. **Conditional Routing gamit ang Mga Desisyon ng Tao**
Maaari kang gumawa ng mga condition function na sumusuri sa mga tugon ng tao:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Mga Real-World Application:

1. **Approval Workflows**
   - Kunin ang pag-apruba ng manager bago iproseso ang mga ulat ng gastusin
   - Kailangan ng pagsusuri ng tao bago magpadala ng automated na mga email
   - Kumpirmahin ang mga transaksyong may mataas na halaga bago isagawa

2. **Content Moderation**
   - I-flag ang mga kuwestiyunable na nilalaman para sa pagsusuri ng tao
   - Hilingin sa mga moderator na gumawa ng huling desisyon sa mga edge case
   - I-escalate sa mga tao kapag mababa ang kumpiyansa ng AI

3. **Customer Service**
   - Hayaan ang AI na hawakan ang mga pangkaraniwang tanong nang awtomatiko
   - I-escalate ang mga komplikadong isyu sa mga human agent
   - Tanungin ang customer kung nais nilang makipag-usap sa isang tao

4. **Data Processing**
   - Hilingin sa mga tao na lutasin ang mga malabong data entry
   - Kumpirmahin ang mga interpretasyon ng AI sa mga hindi malinaw na dokumento
   - Bigyan ng pagpipilian ang mga user sa pagitan ng maraming valid na interpretasyon

5. **Safety-Critical Systems**
   - Kailangan ng kumpirmasyon ng tao bago ang mga hindi na mababalik na aksyon
   - Kunin ang pag-apruba bago i-access ang sensitibong data
   - Kumpirmahin ang mga desisyon sa mga regulated na industriya (healthcare, finance)

6. **Interactive Agents**
   - Gumawa ng mga conversational bot na nagtatanong ng follow-up na mga tanong
   - Gumawa ng mga wizard na gumagabay sa mga user sa mga komplikadong proseso
   - Disenyuhin ang mga agent na nakikipagtulungan sa mga tao step-by-step

### 🔄 Paghahambing: May Human-in-the-Loop vs Wala

| Tampok | Conditional Workflow | Human-in-the-Loop Workflow |
|---------|---------------------|---------------------------|
| **Execution** | Isang `workflow.run()` | Loop gamit ang `run_stream()` + `send_responses_streaming()` |
| **User Input** | Wala (ganap na awtomatiko) | Interactive prompts gamit ang `input()` o UI |
| **Mga Komponent** | Agents + Executors | + RequestInfoExecutor + DecisionManager |
| **Mga Event** | AgentExecutorResponse lamang | RequestInfoEvent, WorkflowOutputEvent, atbp. |
| **Pag-pause** | Walang pag-pause | Naghihinto ang workflow sa RequestInfoExecutor |
| **Kontrol ng Tao** | Walang kontrol ng tao | Ang mga tao ang gumagawa ng mga pangunahing desisyon |
| **Use Case** | Awtomasyon | Kolaborasyon at pag-ooversight |

### 🚀 Advanced na Mga Pattern:

#### Maramihang Punto ng Desisyon ng Tao
Maaari kang magkaroon ng maramihang `RequestInfoExecutor` nodes sa parehong workflow:
```python
.add_edge(agent1, request_info_1)  # Unang desisyon ng tao
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Pangalawang desisyon ng tao
.add_edge(decision_manager_2, final_agent)
```

#### Timeout Handling
Magpatupad ng timeout para sa mga tugon ng tao:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Default sa ligtas na pagpipilian
```

#### Mahusay na UI Integration
Sa halip na `input()`, i-integrate sa web UI, Slack, Teams, atbp.:
```python
if isinstance(event, RequestInfoEvent):
    # Magpadala ng paunawa sa paboritong channel ng gumagamit
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Kondisyong Human-in-the-Loop
Humiling lang ng input ng tao sa mga espesipikong sitwasyon:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # I-route lamang sa tao kung mababa ang kumpiyansa o mataas ang halaga
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Pinakamahusay na Praktis:

1. **Laging Mag-subclass ng RequestInfoMessage**
   - Nagbibigay ng type safety at validation
   - Nagbibigay ng mayamang context para sa UI rendering
   - Nililinaw ang layunin ng bawat uri ng request

2. **Gumamit ng Deskriptibong Mga Prompt**
   - Isama ang konteksto tungkol sa iyong tinatanong
   - Ipaliwanag ang mga kahihinatnan ng bawat pagpipilian
   - Panatilihing simple at malinaw ang mga tanong

3. **Hawakan ang Hindi Inaasahang Input**
   - I-validate ang mga tugon ng user
   - Magbigay ng default para sa maling input
   - Magbigay ng malinaw na mga mensahe ng error

4. **Subaybayan ang Request ID**
   - Gamitin ang korelasyon sa pagitan ng request_id at mga tugon
   - Huwag subukang i-manage ang estado nang mano-mano

5. **Disenyuhin para sa Non-Blocking**
   - Huwag i-block ang mga thread habang naghihintay ng input
   - Gumamit ng mga async pattern sa buong proseso
   - Suportahan ang magkakasabay na workflow instances

### 📚 Kaugnay na Mga Konsepto:

- **Agent Middleware** - Saluhin ang mga tawag ng agent (nakaraang notebook)
- **Workflow State Management** - I-persist ang estado ng workflow sa pagitan ng mga takbo
- **Multi-Agent Collaboration** - Pagsamahin ang human-in-the-loop sa mga team ng agent
- **Event-Driven Architectures** - Bumuo ng mga reaktibong sistema gamit ang mga event

---

### 🎓 Maligayang Pagbati!

Naitaguyod mo na ang mga human-in-the-loop workflow gamit ang Microsoft Agent Framework! Ngayon ay alam mo na kung paano:
- ✅ Pahintuin ang mga workflow para mangolekta ng input ng tao
- ✅ Gamitin ang RequestInfoExecutor at RequestInfoMessage
- ✅ Pamahalaan ang streaming execution gamit ang mga event
- ✅ Gumawa ng custom executors gamit ang @handler
- ✅ I-route ang mga workflow base sa mga desisyon ng tao
- ✅ Bumuo ng interactive AI agent na nakikipagtulungan sa mga tao

**Ito ay isang kritikal na pattern para sa pagbuo ng maaasahan at kontroladong AI system!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
